# Abalone EDA

Exploratory analysis of the UCI Abalone dataset (4,177 rows) that motivates the feature engineering and the `log1p(rings)` training target used in `src/abalone`.

Run from the repository root after `pip install -e ".[all]"`; the first run downloads and caches the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from abalone.config import load_config
from abalone.data import FEATURES, NUMERIC, load_dataset
from abalone.features import MorphometricFeatures
from abalone.validation import validate

cfg = load_config("../configs/default.yaml", overrides={"data": {"cache_dir": "../data/raw"}})
df = load_dataset(cfg.data)
df.head()

## Schema checks and known anomalies

In [ ]:
report = validate(df)
print("rows:", report.n_rows)
report.warnings

In [ ]:
df.describe().T

## Target: ring count

Rings are right-skewed; `log1p` makes the distribution much closer to symmetric, and squared error on `log1p(rings)` is exactly the (squared) RMSLE objective.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
df["rings"].plot.hist(bins=29, ax=axes[0], title="rings")
np.log1p(df["rings"]).plot.hist(bins=29, ax=axes[1], title="log1p(rings)")
print("skew raw: {:.2f}  skew log1p: {:.2f}".format(df["rings"].skew(), np.log1p(df["rings"]).skew()))

## Rings by sex

Infants are clearly younger; one-hot `sex` is informative.

In [ ]:
df.groupby("sex")["rings"].describe()

## Correlations with the target

All size/weight measurements are strongly collinear; shell weight is the single strongest predictor.

In [ ]:
corr = df[[*NUMERIC, "rings"]].corr(method="spearman")
corr["rings"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(label="Spearman rho")
plt.tight_layout()

## Engineered features

Ratios (body composition), volume/density and log weights add signal that is not linear in the raw columns.

In [ ]:
feats = MorphometricFeatures().fit_transform(df[FEATURES])
eng = feats.drop(columns=FEATURES).assign(log_rings=np.log1p(df["rings"]))
eng.corr(method="spearman")["log_rings"].drop("log_rings").sort_values(key=abs, ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for sex, g in df.groupby("sex"):
    ax.scatter(g["shell_weight"], g["rings"], s=6, alpha=0.4, label=sex)
ax.set(xlabel="shell weight", ylabel="rings", title="Shell weight vs rings")
ax.legend()